In [1]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.sql.functions import col, sum, udf, when
from pyspark.sql.types import IntegerType
from pyspark.sql import Row


FEATURE_COLS = [
    "amt",
    "log_amt",
    "city_pop",
    "customer_age",
    "distance_customer_merchant",
    "trans_hour",
    "trans_dayofweek",
    "trans_month",
    "category_index"
]

StatementMeta(, f63d5883-9c11-4f08-83a9-fe337d16f77b, 3, Finished, Available, Finished)

### Load ML tables and change feature column types

In [2]:
df_train = spark.sql("SELECT * FROM fraud_detection_lakehouse.dbo.ml_train_features")
df_test = spark.sql("SELECT * FROM fraud_detection_lakehouse.dbo.ml_test_features")

df_train = df_train.select(
    "trans_num",
    "trans_date_trans_time",
    "cc_num",
    *[col(c).cast("double").alias(c) for c in FEATURE_COLS],
    col("is_fraud").cast("double")
).dropna()

df_test = df_test.select(
    "trans_num",
    "trans_date_trans_time",
    "cc_num",
    *[col(c).cast("double").alias(c) for c in FEATURE_COLS],
    col("is_fraud").cast("double")
).dropna()

StatementMeta(, f63d5883-9c11-4f08-83a9-fe337d16f77b, 4, Finished, Available, Finished)

### Build vectors

In [3]:
assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features")

train_vec = assembler.transform(df_train).select("features", "is_fraud")
test_vec = assembler.transform(df_test).select("features", "is_fraud")

StatementMeta(, f63d5883-9c11-4f08-83a9-fe337d16f77b, 5, Finished, Available, Finished)

### Create Logistic Regression model

In [4]:
lr = LogisticRegression(featuresCol="features", labelCol="is_fraud")

StatementMeta(, f63d5883-9c11-4f08-83a9-fe337d16f77b, 6, Finished, Available, Finished)

### Train model

In [5]:
lr_model = lr.fit(train_vec)

StatementMeta(, f63d5883-9c11-4f08-83a9-fe337d16f77b, 7, Finished, Available, Finished)

### Predict on Test

In [6]:
predictions = lr_model.transform(test_vec)
predictions.select("is_fraud", "probability", "prediction").show(5)

StatementMeta(, f63d5883-9c11-4f08-83a9-fe337d16f77b, 8, Finished, Available, Finished)

+--------+--------------------+----------+
|is_fraud|         probability|prediction|
+--------+--------------------+----------+
|     0.0|[0.99995201937466...|       0.0|
|     0.0|[0.99779780628653...|       0.0|
|     0.0|[0.99564551653040...|       0.0|
|     0.0|[0.99984234243485...|       0.0|
|     0.0|[0.99872353534840...|       0.0|
+--------+--------------------+----------+
only showing top 5 rows



### Treshold logic

In [7]:
THRESHOLD = 0.25

get_fraud_prob = udf(lambda v: float(v[1]), "double")
predictions = predictions.withColumn(
    "fraud_probability", 
    get_fraud_prob(col("probability"))
)
predictions = predictions.withColumn(
    "fraud_pred_custom",
    (col("fraud_probability") >= THRESHOLD).cast("int")
)

StatementMeta(, f63d5883-9c11-4f08-83a9-fe337d16f77b, 9, Finished, Available, Finished)

### Confusion Metrics

In [8]:
cm = predictions.select(
    sum(when((col("is_fraud") == 1) & (col("fraud_pred_custom") == 1), 1)).alias("TP"),
    sum(when((col("is_fraud") == 0) & (col("fraud_pred_custom") == 1), 1)).alias("FP"),
    sum(when((col("is_fraud") == 1) & (col("fraud_pred_custom") == 0), 1)).alias("FN"),
    sum(when((col("is_fraud") == 0) & (col("fraud_pred_custom") == 0), 1)).alias("TN")
)

cm.show()

StatementMeta(, f63d5883-9c11-4f08-83a9-fe337d16f77b, 10, Finished, Available, Finished)

+---+---+----+------+
| TP| FP|  FN|    TN|
+---+---+----+------+
|208|576|1937|552998|
+---+---+----+------+



### Measure Metrics

In [9]:
metrics = cm.collect()[0]

TP = metrics["TP"]  # Fraud
FP = metrics["FP"]  # Blocked normal transaction
FN = metrics["FN"]  # Missed Fraud
TN = metrics["TN"]  # Normal transaction behaviour

recall = TP / (TP + FN)
precision = TP / (TP + FP)

print("Recall:", recall)
print("Precision:", precision)

StatementMeta(, f63d5883-9c11-4f08-83a9-fe337d16f77b, 11, Finished, Available, Finished)

Recall: 0.09696969696969697
Precision: 0.2653061224489796


### Save Metrics

In [10]:
metrics_df = spark.createDataFrame([
    Row(
        model="LogisticRegression",
        threshold=THRESHOLD,
        recall=recall,
        precision=precision,
        TP=TP,
        FP=FP,
        FN=FN,
        TN=TN
    )
])

metrics_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ml_model_metrics")

StatementMeta(, f63d5883-9c11-4f08-83a9-fe337d16f77b, 12, Finished, Available, Finished)

### Save model

In [11]:
lr_model.write().overwrite().save("Files/models/fraud_lr_model")

StatementMeta(, f63d5883-9c11-4f08-83a9-fe337d16f77b, 13, Finished, Available, Finished)